In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import sys
from pathlib import Path
# metrics.py lives at ../../../src/metrics/metrics.py relative to this notebook
sys.path.insert(0, str((Path("../../../src/metrics")).resolve()))
from metrics import (
    organize_multiverse_results,
    within_bucket_variation,
    total_variance_decomposition,
)

# get all csv files in the data directory for run of interest
data_dir = Path("../output/runs/1/data")
csv_files = list(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files.")

# read all files with mixed types
arrays_lst = []
arrays = []
for i, f in enumerate(csv_files):
    arr = np.genfromtxt(f, delimiter=',', dtype=None, encoding='utf-8')
    arrays.append(arr)
    if (i + 1) % 1000 == 0:
        arrays_lst.append(np.concatenate(arrays))
        arrays = []
        print(f"Loaded {i+1}/{len(csv_files)} files")
arrays_lst.append(np.concatenate(arrays))
# stack all arrays vertically
data = np.concatenate(arrays_lst)
# convert data to pandas df
df = pd.DataFrame(data)
# remove all rows that are exact duplicates, since they are removing the rows of column names
df = df.drop_duplicates()
# make first row the header
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

In [ ]:
# look at data
print(df.shape)
df.head()

In [ ]:
# set up inputs for `organize_multiverse_results`
dimension_order = ["spell_gap_threshold",
 "new_job_notif_threshold",
 "new_job_no_notif_threshold",
 "missing_strategy",
 "split_seed",
 "model"]
# prepend mv_dim_ to all dimension names
dimension_order = ["mv_dim_" + dim for dim in dimension_order]
performance_metrics = ["acc", "f1", "roc_auc"]
# prepend test_ to all performance metric names
performance_metrics = ["test_" + metric for metric in performance_metrics]

# make all cols that start with test_ numeric
for col in df.columns:
    if col.startswith("test_"):
        df[col] = pd.to_numeric(df[col])

In [ ]:
# call helper function to make a structured df
df = organize_multiverse_results(df, dimension_order, performance_metrics)

In [ ]:
# ensure types are correct
df.dtypes

In [ ]:
variance_decomposition = total_variance_decomposition(df, "test_acc")
# BUV is 'expected_var_within_groups'
buv = {key: variance_decomposition[key]['expected_var_within_groups'] for key in variance_decomposition.keys()}

In [ ]:
wuv = within_bucket_variation(df, "test_acc")

In [ ]:
# make regression predicting test_acc from all dimensions, and get coefficients and r^2
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
# get all dimension cols
dimension_cols = [col for col in df.columns if col[0].startswith("mv_dim_")]
X = df[dimension_cols]
# get dummy variables for dimensions that are categorical
X = pd.get_dummies(X, columns=X.columns)
y = df["test_acc"].astype(float)
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
coefficients = dict(zip(dimension_cols, model.coef_))
print("R^2:", r2)
print("Coefficients:", coefficients)